In [2]:
import numpy as np
import pandas as pd
import random
import torch
import json
from transformers import BertTokenizer, BertModel
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

c:\Users\lppoh\anaconda3\envs\NLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Set a random seed for reproducibility
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

In [4]:
def embed_words(words):
    """
    Embeds a list of words using the BERT model.

    Args:
        words (list): List of words or sentences to embed.

    Returns:
        torch.Tensor: The average embedding of each input.
    """
    encoding = tokenizer.batch_encode_plus(
        words,                    # List of input texts
        padding=True,             # Pad to the maximum sequence length
        truncation=True,          # Truncate to the maximum sequence length if necessary
        return_tensors='pt',      # Return PyTorch tensors
        add_special_tokens=True   # Add special tokens CLS and SEP
    )
    
    input_ids = encoding['input_ids']
    attention_mask = encoding['attention_mask']

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        embeddings = outputs.last_hidden_state.mean(dim=1)  # Average over the sequence length
    return embeddings

In [5]:
# Load SBIC
SB = pd.read_csv('../../input/data/SBIC.v2.agg.trn.csv')

In [6]:
#Process SBIC
SB['id'] = ['SB' + str(i) for i in range(1, len(SB) + 1)]
SB.set_index(['id'], inplace=True)
SB = SB.rename(columns={'post': 'text'})

SB['true_label'] = [1 if SB['offensiveYN'].iloc[i] >= 0.5 or SB['intentYN'].iloc[i] else 0 for i in range(len(SB))]

In [7]:
# Process Words to list
SB['targetMinority'] = SB['targetMinority'].apply(json.loads)
words_list = [SB['targetMinority'].iloc[i] for i in range(len(SB['targetMinority'])) if SB['targetMinority'].iloc[i] != []]
words = [word for sub in words_list for word in sub]

#Calculate coresponding id
id = [SB.index[i] for i in range(len(SB['targetMinority'])) if SB['targetMinority'].iloc[i] != []for j in range(len(SB['targetMinority'].iloc[i]))]

In [44]:
#Unpack strings further 
words_list = []
id_list = []
for i in range(len(words)):
    w = words[i]
    w = w.replace(', ', ' ')
    w = w.replace('/', ' ')
    if ' ' in w:
        w = w.split(' ')
        for j in range(len(w)):
            words_list.append(w[j])
            id_list.append(id[i])
    else:
        words_list.append(w)
        id_list.append(id[i])

In [47]:
# Calculate word embeddings
embeddings = embed_words(words_list)

group_mapping = ['asian', 'muslim', 'jewish','black','lgbtq','woman']
group_embeddings = embed_words(group_mapping)

In [70]:
clustered_ids = {}
for i in range(len(id_list)):
    for j in range(5):
        cos = cosine_similarity(embeddings[i].reshape(1,-1), group_embeddings[j].reshape(1,-1))
        if cos >= 0.85:# and id[i] not in clustered_ids:
            clustered_ids[id_list[i]] = group_mapping[j]

In [82]:
#Safe SBIC Group
SB['target']= SB.index.map(clustered_ids)
SB_group= SB.sort_values(by = 'target')[['text', 'true_label', 'target']]
SB_group.to_csv("../../input/data/SBIC_group.csv", index=True)

In [83]:
#Safe SBIC Aggregate
SB_aggregate = SB.sort_values('true_label')[['text', 'true_label']]
SB_aggregate.to_csv("../../input/data/SBIC_aggregate.csv", index=True)